# Script 2 — Fine-Tuning Mistral-7B-Instruct · Venous Medicine
**Hardware:** RTX 5090 (32 GB VRAM) · **MLOps:** Weights & Biases

### Methods active in this notebook
| Goal | Method | Key flag |
|---|---|---|
| Faster training (2–2.7×) | **Unsloth** fused Triton kernels | `FastLanguageModel` |
| Memory + base compression | **QLoRA** 4-bit NF4 + double quant | `load_in_4bit=True` |
| Proper weight init | **PiSSA** SVD-based adapter init | `init_lora_weights="pissa"` |
| Better convergence | **DoRA** magnitude + direction decomp | `use_dora=True` |
| Stable high-rank gradients | **rsLoRA** `alpha/√r` scaling | `use_rslora=True` |
| Best adaptation quality | **All-linear** target modules | `target_modules="all-linear"` |
| Eliminate padding waste | **Packing** short sequences together | `packing=True` |

Run cells top-to-bottom. W&B dashboard opens after Cell 10 starts training.

In [1]:
# ── 0. Install dependencies ───────────────────────────────────────────────────
import subprocess, sys, platform

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# PyTorch is pre-installed in venv. Only install extra deps.
pip('transformers>=4.40', 'datasets>=2.19', 'peft>=0.10', 'trl>=0.8',
    'bitsandbytes>=0.43', 'accelerate>=0.29', 'wandb>=0.17',
    'scipy', 'sentencepiece')

# Flash Attention 2 (RTX 5090 supports it; skip if build fails)
try:
    pip('flash-attn', '--no-build-isolation')
    FLASH_ATTN = True
    print('[INFO] Flash Attention 2 installed.')
except Exception as e:
    print(f'[INFO] flash-attn build failed ({e}) — continuing without it.')
    FLASH_ATTN = False

# Unsloth (requires Linux/WSL; native Windows falls back to PEFT)
try:
    pip('unsloth')
    print('[INFO] Unsloth installed.')
except Exception:
    print('[INFO] Unsloth install skipped — will use PEFT path.')

import torch
assert torch.cuda.is_available(), 'CUDA not available — check CUDA Toolkit installation'
print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'OS   : {platform.system()}')

[INFO] flash-attn build failed (Command '['c:\\Users\\Krish\\Downloads\\LLM_Finetuning\\venv_pytorch\\Scripts\\python.exe', '-m', 'pip', 'install', '-q', 'flash-attn', '--no-build-isolation']' returned non-zero exit status 1.) — continuing without it.
[INFO] Unsloth installed.


AssertionError: CUDA not available — check CUDA Toolkit installation

In [4]:
! pip freeze

accelerate==1.13.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.5
aiosignal==1.4.0
annotated-doc==0.0.4
annotated-types==0.7.0
anyio==4.13.0
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
arrow==1.4.0
asttokens==3.0.1
async-lru==2.3.0
attrs==26.1.0
babel==2.18.0
beautifulsoup4==4.14.3
bitsandbytes==0.49.2
bleach==6.3.0
certifi==2026.4.22
cffi==2.0.0
charset-normalizer==3.4.7
click==8.3.3
colorama==0.4.6
comm==0.2.3
cut-cross-entropy==25.1.1
datasets==4.3.0
debugpy==1.8.20
decorator==5.2.1
defusedxml==0.7.1
diffusers==0.37.1
dill==0.4.0
docstring_parser==0.18.0
executing==2.2.1
fastjsonschema==2.21.2
filelock==3.25.2
fqdn==1.5.1
frozenlist==1.8.0
fsspec==2025.9.0
gitdb==4.0.12
GitPython==3.1.49
h11==0.16.0
hf-xet==1.4.3
hf_transfer==0.1.9
httpcore==1.0.9
httpx==0.28.1
huggingface_hub==1.12.2
idna==3.13
importlib_metadata==9.0.0
ipykernel==7.2.0
ipython==9.13.0
ipython_pygments_lexers==1.1.1
ipywidgets==8.1.8
isoduration==20.11.0
jedi==0.19.2
Jinja2==3.1.6
json5==0.14.0
jsonpointer==3.1.1
jso

In [5]:
! nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Tue_Feb_27_16:28:36_Pacific_Standard_Time_2024
Cuda compilation tools, release 12.4, V12.4.99
Build cuda_12.4.r12.4/compiler.33961263_0


In [ ]:
! source ~/pytorch_env/bin/activate
! python -c "import torch; print(f'CUDA: {torch.cuda.is_available()}'); print(f'PyTorch: {torch.__version__}'); print(f'CUDA version: {torch.version.cuda}')"

'source' is not recognized as an internal or external command,
operable program or batch file.


CUDA: False
PyTorch: 2.10.0+cpu
CUDA version: None


In [ ]:
# ── 1. W&B login ──────────────────────────────────────────────────────────────
import wandb, os

wandb.login()   # first run: browser prompt; subsequent runs: cached key

WANDB_PROJECT  = 'mistral-venous-medicine'
WANDB_RUN_NAME = 'mistral7b-pissa-dora-rslora-unsloth-v1'
os.environ['WANDB_PROJECT'] = WANDB_PROJECT
print(f'W&B project: {WANDB_PROJECT}')

In [ ]:
# ── 2. Config ─────────────────────────────────────────────────────────────────
from pathlib import Path

BASE_MODEL_ID    = 'mistralai/Mistral-7B-Instruct-v0.2'
TRAIN_FILE       = Path('data/training_data_train.jsonl')
VAL_FILE         = Path('data/training_data_val.jsonl')
OUTPUT_DIR       = Path('training_output')
LORA_ADAPTER_DIR = Path('lora_adapter')
MERGED_MODEL_DIR = Path('merged_model')

# ── Training hyperparameters
NUM_EPOCHS       = 3
BATCH_SIZE       = 4       # RTX 5090 32GB handles 4 comfortably with 4-bit
GRAD_ACCUM_STEPS = 4       # effective batch = 16
LEARNING_RATE    = 2e-4    # LoRA sweet-spot; cosine scheduler decays from here
MAX_SEQ_LENGTH   = 2048
WARMUP_RATIO     = 0.05
LR_SCHEDULER     = 'cosine'
SAVE_STEPS       = 100
EVAL_STEPS       = 100
LOGGING_STEPS    = 10
RANDOM_SEED      = 42

# ── LoRA / adapter config
LORA_R       = 32
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LORA_TARGET  = 'all-linear'

# ── Method flags
USE_DORA    = True
USE_RSLORA  = True
LORA_INIT   = 'pissa'

print('Config loaded.')
print(f'  r={LORA_R}, alpha={LORA_ALPHA}, init={LORA_INIT}, DoRA={USE_DORA}, rsLoRA={USE_RSLORA}')
print(f'  Target modules: {LORA_TARGET}')

In [ ]:
# ── 3. Load dataset ───────────────────────────────────────────────────────────
from datasets import load_dataset

ds = load_dataset('json',
                  data_files={'train': str(TRAIN_FILE), 'validation': str(VAL_FILE)})
print(ds)
print('\nSample (first 300 chars):')
print(ds['train'][0]['text'][:300], '...')

In [ ]:
# ── 4. Detect Unsloth availability and set runtime path ───────────────────────
USE_UNSLOTH = False
try:
    from unsloth import FastLanguageModel
    from unsloth.training_args import UnslothTrainingArguments
    USE_UNSLOTH = True
    print('Unsloth detected — using fused Triton kernels (2–2.7× faster).')
except ImportError:
    print('Unsloth not available — using standard PEFT + HuggingFace Trainer.')
    print('Training will be correct but without kernel-level speed boost.')

print(f'Runtime: {"Unsloth" if USE_UNSLOTH else "PEFT"}')

In [ ]:
# ── 5. Load base model + tokenizer ───────────────────────────────────────────
import torch

if USE_UNSLOTH:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None,
        device_map='auto',
    )
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    attn_impl = 'flash_attention_2' if FLASH_ATTN else 'eager'
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        torch_dtype=torch.bfloat16,
        attn_implementation=attn_impl,
    )
    model.config.use_cache = False
    model.config.pretraining_tp = 1

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

total_params = sum(p.numel() for p in model.parameters())
print(f'Base model loaded: {total_params / 1e9:.2f}B params')
print(f'Tokenizer vocab size: {tokenizer.vocab_size}')

In [ ]:
# ── 6. Apply LoRA adapters: PiSSA + DoRA + rsLoRA + all-linear ────────────────
if USE_UNSLOTH:
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=RANDOM_SEED,
        use_dora=USE_DORA,
        use_rslora=USE_RSLORA,
        init_lora_weights=LORA_INIT,
    )
else:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET,
        bias='none',
        task_type='CAUSAL_LM',
        use_dora=USE_DORA,
        use_rslora=USE_RSLORA,
        init_lora_weights=LORA_INIT,
    )
    model = get_peft_model(model, lora_config)
    model.enable_input_require_grads()

model.print_trainable_parameters()
print(f'\nMethod summary:')
print(f'  Init   : {LORA_INIT.upper()} (SVD-based, aligned with principal weight directions)')
print(f'  DoRA   : {USE_DORA} (magnitude + direction decomposition)')
print(f'  rsLoRA : {USE_RSLORA} (alpha/sqrt(r) = {LORA_ALPHA}/{LORA_R**0.5:.2f} ≈ {LORA_ALPHA/LORA_R**0.5:.2f})')
print(f'  Target : {LORA_TARGET}')

In [ ]:
# ── 7. Training arguments ─────────────────────────────────────────────────────
from transformers import TrainingArguments

OPTIM = 'adamw_8bit' if USE_UNSLOTH else 'paged_adamw_32bit'
GC = not USE_UNSLOTH

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=GC,
    optim=OPTIM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    fp16=False,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='wandb',
    run_name=WANDB_RUN_NAME,
    group_by_length=True,
    dataloader_pin_memory=True,
    seed=RANDOM_SEED,
)

eff_batch = BATCH_SIZE * GRAD_ACCUM_STEPS
steps_per_epoch = len(ds['train']) // eff_batch
print(f'Effective batch size : {eff_batch}')
print(f'Steps / epoch        : ~{steps_per_epoch}')
print(f'Optimizer            : {OPTIM}')
print(f'Gradient checkpointing: {"Unsloth" if USE_UNSLOTH else "HuggingFace"}')

In [ ]:
# ── 8. SFT Trainer ────────────────────────────────────────────────────────────
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    tokenizer=tokenizer,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
)
print('Trainer ready.')
print(f'Train examples : {len(ds["train"]):,}')
print(f'Val   examples : {len(ds["validation"]):,}')

In [ ]:
# ── 9. Log hyperparameters to W&B before training starts ─────────────────────
wandb.config.update({
    'base_model'      : BASE_MODEL_ID,
    'lora_r'          : LORA_R,
    'lora_alpha'      : LORA_ALPHA,
    'lora_dropout'    : LORA_DROPOUT,
    'lora_target'     : LORA_TARGET,
    'lora_init'       : LORA_INIT,
    'use_dora'        : USE_DORA,
    'use_rslora'      : USE_RSLORA,
    'effective_scale' : round(LORA_ALPHA / LORA_R**0.5, 4),
    'use_unsloth'     : USE_UNSLOTH,
    'quantization'    : '4-bit NF4 + double quant',
    'optimizer'       : OPTIM,
    'learning_rate'   : LEARNING_RATE,
    'epochs'          : NUM_EPOCHS,
    'batch_size_eff'  : BATCH_SIZE * GRAD_ACCUM_STEPS,
    'max_seq_length'  : MAX_SEQ_LENGTH,
    'packing'         : True,
    'train_examples'  : len(ds['train']),
    'val_examples'    : len(ds['validation']),
})
print('Hyperparameters logged to W&B.')

In [ ]:
# ── 10. Train ─────────────────────────────────────────────────────────────────
train_result = trainer.train()

print('\n=== Training complete ===')
print(f'Final train loss : {train_result.training_loss:.4f}')
print(f'Total steps      : {train_result.global_step}')

wandb.log({'train/final_loss': train_result.training_loss,
           'train/total_steps': train_result.global_step})

In [ ]:
# ── 11. Validation loss + perplexity ─────────────────────────────────────────
import math

eval_metrics = trainer.evaluate(eval_dataset=ds['validation'])

val_loss    = eval_metrics['eval_loss']
perplexity  = math.exp(val_loss)

print(f'Validation loss       : {val_loss:.4f}')
print(f'Validation perplexity : {perplexity:.2f}')

wandb.log({
    'final/val_loss'       : val_loss,
    'final/val_perplexity' : perplexity,
    'final/train_loss'     : train_result.training_loss,
})
print('Metrics logged to W&B.')

In [ ]:
# ── 12. Token-level accuracy on validation set ────────────────────────────────
import torch

if USE_UNSLOTH:
    FastLanguageModel.for_inference(model)

model.eval()
correct_tokens, total_tokens = 0, 0
n_samples = min(200, len(ds['validation']))

with torch.no_grad():
    for i in range(n_samples):
        text   = ds['validation'][i]['text']
        inputs = tokenizer(text, return_tensors='pt',
                           truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
        labels = inputs['input_ids'].clone()
        out    = model(**inputs, labels=labels)
        logits  = out.logits[:, :-1, :]
        targets = labels[:, 1:]
        preds   = logits.argmax(dim=-1)
        mask    = targets != tokenizer.pad_token_id
        correct_tokens += (preds == targets)[mask].sum().item()
        total_tokens   += mask.sum().item()

token_accuracy = correct_tokens / total_tokens if total_tokens else 0.0
print(f'Token-level accuracy ({n_samples} val samples): {token_accuracy:.4f}  ({token_accuracy*100:.2f}%)')

wandb.log({'final/val_token_accuracy': token_accuracy})

In [ ]:
# ── 13. Save LoRA adapter ─────────────────────────────────────────────────────
LORA_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

if USE_UNSLOTH:
    model.save_pretrained(str(LORA_ADAPTER_DIR))
    tokenizer.save_pretrained(str(LORA_ADAPTER_DIR))
else:
    trainer.model.save_pretrained(str(LORA_ADAPTER_DIR))
    tokenizer.save_pretrained(str(LORA_ADAPTER_DIR))

print(f'LoRA adapter (PiSSA + DoRA + rsLoRA) saved → {LORA_ADAPTER_DIR}')

In [ ]:
# ── 14. Merge LoRA into base model and save ───────────────────────────────────
MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

if USE_UNSLOTH:
    model.save_pretrained_merged(
        str(MERGED_MODEL_DIR),
        tokenizer,
        save_method='merged_16bit',
    )
else:
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM

    print('Loading base model in fp16 for merge (no quantisation)...')
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float16,
        device_map='auto',
    )
    print('Merging...')
    merged = PeftModel.from_pretrained(base, str(LORA_ADAPTER_DIR))
    merged = merged.merge_and_unload()
    merged.save_pretrained(str(MERGED_MODEL_DIR), safe_serialization=True)
    tokenizer.save_pretrained(str(MERGED_MODEL_DIR))

print(f'Merged model saved → {MERGED_MODEL_DIR.resolve()}')

In [ ]:
# ── 15. Finalise W&B run ──────────────────────────────────────────────────────
wandb.summary.update({
    'base_model'          : BASE_MODEL_ID,
    'lora_r'              : LORA_R,
    'lora_alpha'          : LORA_ALPHA,
    'lora_init'           : LORA_INIT,
    'use_dora'            : USE_DORA,
    'use_rslora'          : USE_RSLORA,
    'rslora_effective_scale': round(LORA_ALPHA / LORA_R**0.5, 4),
    'use_unsloth'         : USE_UNSLOTH,
    'train_loss'          : train_result.training_loss,
    'val_loss'            : val_loss,
    'val_perplexity'      : perplexity,
    'val_token_accuracy'  : token_accuracy,
    'epochs'              : NUM_EPOCHS,
    'merged_model_path'   : str(MERGED_MODEL_DIR.resolve()),
})
wandb.finish()

print('=== All done ===')
print(f'W&B run finished. Dashboard: https://wandb.ai/{wandb.run.entity}/{WANDB_PROJECT}')
print(f'Merged model : {MERGED_MODEL_DIR.resolve()}')
print(f'LoRA adapter : {LORA_ADAPTER_DIR.resolve()}')